In [4]:
import pandas as pd

# Load data
true = pd.read_csv("/kaggle/input/datasets/utkarshsrivastavji/fake-and-true-dataset-for-lstm-model-prediction/Fake.csv")
fake = pd.read_csv("/kaggle/input/datasets/utkarshsrivastavji/fake-and-true-dataset-for-lstm-model-prediction/True.csv")

# Add labels
fake['label'] = 0   # fake news
true['label'] = 1   # real news

# Combine
df = pd.concat([fake, true], ignore_index=True)

# Shuffle
df = df.sample(frac=1).reset_index(drop=True)

# Combine text columns (important for LSTM)
df['content'] = df['title'] + " " + df['text']

# Keep only needed columns
df = df[['content', 'label']]

# Drop null
df = df.dropna()

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)   # remove extra spaces
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text.strip()

df['content'] = df['content'].apply(clean_text)

df = df.drop_duplicates(subset='content')

# Check
print(df.head(5))
print(df['label'].value_counts())



                                             content  label
0  awesome new app helps make sure we all continu...      1
1  china lodges stern protest with south korea ov...      0
2  zimbabwes mugabe likens rivals to judas for se...      0
3  why democrats can thank harry reid for replaci...      1
4  marco rubio emerges as champion of battered re...      0
label
0    21178
1    17905
Name: count, dtype: int64


In [5]:
df.isnull().sum()


content    0
label      0
dtype: int64

In [6]:
X=df['content']
Y=df['label']

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,shuffle=True,stratify=Y)




In [7]:

from tensorflow.keras.preprocessing.text import Tokenizer
token = Tokenizer(num_words=20000, oov_token="<OOV>")
token.fit_on_texts(X_train)



/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [10]:
X_train_seq = token.texts_to_sequences(X_train)
X_test_seq = token.texts_to_sequences(X_test)


In [11]:
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences
lengths = [len(seq) for seq in X_train_seq]
print("Max:", max(lengths))
print("Mean:", np.mean(lengths))
print("95 percentile:", np.percentile(lengths, 95))

Max: 8135
Mean: 408.671272308578
95 percentile: 887.0


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=1000, padding='pre')
X_test_pad = pad_sequences(X_test_seq, maxlen=1000, padding='pre')


In [14]:
!pip install keras-tuner
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
def build_model(hp):
    model = Sequential()

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256]),
    ))

    model.add(LSTM(
        units=hp.Choice('lstm_units', [32, 64, 128])
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(
            learning_rate=hp.Choice('lr', [1e-3, 5e-4, 1e-4])
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [16]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_utkaR',
    project_name='fake_news'
)


2026-02-25 15:12:50.616814: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
tuner.search(
    X_train_pad,
    Y_train,
    epochs=3,
    validation_split=0.2
)

Trial 5 Complete [00h 18m 35s]
val_accuracy: 0.988007664680481

Best val_accuracy So Far: 0.988007664680481
Total elapsed time: 01h 09m 40s


In [17]:
#best_model = tuner.get_best_models(1)[0]
#best_model.summary()    DONE AND THE BEST PARAMS ARE embedding_dim = 64
#lstm_units = 128
#dropout = 0.3
#lr = 0.0005



IndexError: list index out of range

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=64, input_length=540))

# LSTM
model.add(LSTM(128, dropout=0.3, recurrent_dropout=0.3))

# Dense
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

In [14]:
model = tuner.get_best_models(1)[0]

loss, acc = model.evaluate(X_test_pad, Y_test)
print("Test Accuracy:", acc)

245/245 ━━━━━━━━━━━━━━━━━━━━ 44s 179ms/step - accuracy: 0.9876 - loss: 0.0429
Test Accuracy: 0.9875911474227905


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=128, input_length=540))

# Single strong BiLSTM (faster + stable)
model.add(Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)))

# Dense layer (feature extraction)
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

In [21]:
from tensorflow.keras.optimizers import Adam, RMSprop

def build_model(hp):
    model = Sequential()

    model.add(Input(shape=(500,)))

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256])
    ))

    model.add(Bidirectional(
        LSTM(hp.Choice('lstm_units', [32, 64, 128]))
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    # 🔥 optimizer tuning
    optimizer_choice = hp.Choice('optimizer', ['adam', 'rmsprop'])

    lr = hp.Choice('lr', [1e-3, 5e-4, 1e-4])

    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [18]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

{'embedding_dim': 64, 'lstm_units': 128, 'dropout': 0.3, 'lr': 0.0005}


In [19]:
best_model = tuner.hypermodel.build(best_hp)

In [22]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    Y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 219s 548ms/step - accuracy: 0.9022 - loss: 0.2385 - val_accuracy: 0.9706 - val_loss: 0.0855
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 211s 540ms/step - accuracy: 0.9765 - loss: 0.0783 - val_accuracy: 0.9699 - val_loss: 0.0890
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 211s 540ms/step - accuracy: 0.9801 - loss: 0.0670 - val_accuracy: 0.9739 - val_loss: 0.0828
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 211s 539ms/step - accuracy: 0.9868 - loss: 0.0435 - val_accuracy: 0.9677 - val_loss: 0.1015
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 210s 537ms/step - accuracy: 0.9917 - loss: 0.0295 - val_accuracy: 0.9735 - val_loss: 0.0966


In [23]:
model.evaluate(X_test_pad, Y_test)

245/245 ━━━━━━━━━━━━━━━━━━━━ 20s 81ms/step - accuracy: 0.9779 - loss: 0.0667


[0.06671755760908127, 0.9778687357902527]

In [23]:
model.save("fake_news_model.h5")

import pickle
with open("token.pkl", "wb") as f:
    pickle.dump(token, f)

In [24]:
import os
os.listdir('/kaggle/working')

['token.pkl', 'fake_news_model.h5', '.virtual_documents', 'tuner_utkaR']

In [28]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

In [30]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_news(text):
    text = clean_text(text)

    seq = token.texts_to_sequences([text])
    print("Sequence:", seq)   # DEBUG

    padded = pad_sequences(seq, maxlen=540)
    print("Padded sum:", padded.sum())  # DEBUG

    pred = model.predict(padded)[0][0]

    threshold = 0.6   # try 0.6 or 0.7

    if pred > threshold:
        print("Fake News")
    else:
        print("Real News")

In [34]:
news = input("Enter news: ")
predict_news(news)

Enter news:  New law passed that allows citizens to stop paying taxes without any penalty


Sequence: [[59, 130, 830, 8, 2307, 624, 3, 402, 1545, 816, 264, 98, 3886]]
Padded sum: 10972
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
Fake News


In [35]:
import pickle

# model save
model.save("/kaggle/working/fake_news_model.h5")

# tokenizer save (tu ne 'token' use kiya tha)
with open("/kaggle/working/token.pkl", "wb") as f:
    pickle.dump(token, f)

In [36]:
import os
os.listdir('/kaggle/working')

['token.pkl', 'fake_news_model.h5', '.virtual_documents', 'tuner_utkaR']